# SASV setup check (ASVspoof 2019 LA)

Confirms paths for **SASV 2022-style** evaluation:

- Official trial protocols from `SASVC2022_Baseline`
- ASVspoof 2019 LA audio + enrolment lists
- `get_all_EERs` metric smoke test

Metrics:

| Metric | Meaning |
|--------|--------|
| **SV-EER** | target vs nontarget |
| **SPF-EER** | target vs spoof |
| **SASV-EER** | target vs (nontarget + spoof) |

Run this notebook first. Then open `02_ecapa_only_sasv.ipynb`.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "experiment_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019")
sys.path.insert(0, str(ROOT))

from experiment_lib import (
    DEFAULT_LA,
    DEFAULT_SASV,
    DEFAULT_SERVER,
    ensure_sasv_on_path,
    read_enroll_map,
    read_trials,
    resolve_audio_path,
    trial_key_counts,
)

print("LA:", DEFAULT_LA, DEFAULT_LA.exists())
print("SASV:", DEFAULT_SASV, DEFAULT_SASV.exists())
print("server:", DEFAULT_SERVER, (DEFAULT_SERVER / "ml_server").exists())

LA: D:\speaker-verification-system\data\LA True
SASV: D:\speaker-verification-system\SASVC2022_Baseline True
server: D:\speaker-verification-system\app\server True


## Protocol + audio sanity

Reads the first few **dev** trials and checks the first test flac exists.

In [2]:
sasv = ensure_sasv_on_path()
trials = read_trials(sasv, "dev", max_trials=20)
print("sample trials:", trial_key_counts(trials))
for t in trials[:5]:
    print(t)

path = resolve_audio_path(DEFAULT_LA, "dev", trials[0].test_utt)
print("first test audio:", path, path.exists())

enroll = read_enroll_map(DEFAULT_LA, "dev")
spk = trials[0].speaker_id
print(f"enrol utts for {spk}:", len(enroll.get(spk, [])), enroll.get(spk, [])[:3])

sample trials: {'target': 20, 'nontarget': 0, 'spoof': 0, 'total': 20}
SasvTrial(speaker_id='LA_0073', test_utt='LA_D_4004968', attack_or_type='bonafide', key='target')
SasvTrial(speaker_id='LA_0073', test_utt='LA_D_6027798', attack_or_type='bonafide', key='target')
SasvTrial(speaker_id='LA_0073', test_utt='LA_D_3986002', attack_or_type='bonafide', key='target')
SasvTrial(speaker_id='LA_0073', test_utt='LA_D_9330492', attack_or_type='bonafide', key='target')
SasvTrial(speaker_id='LA_0073', test_utt='LA_D_1364611', attack_or_type='bonafide', key='target')
first test audio: D:\speaker-verification-system\data\LA\ASVspoof2019_LA_dev\flac\LA_D_4004968.flac True
enrol utts for LA_0073: 19 ['LA_D_A1225110', 'LA_D_A1425742', 'LA_D_A2486210']


## Metric smoke test

Perfectly separated fake scores → all EERs ≈ 0.

In [3]:
from metrics import get_all_EERs

keys = ["target", "target", "nontarget", "spoof"]
preds = [0.9, 0.8, 0.2, 0.1]
sasv_eer, sv_eer, spf_eer = get_all_EERs(preds, keys)
print({"sasv_eer": sasv_eer, "sv_eer": sv_eer, "spf_eer": spf_eer})

{'sasv_eer': 0.0, 'sv_eer': 0.0, 'spf_eer': 0.0}


## Next

Open **`02_ecapa_only_sasv.ipynb`**.

Keep `SMOKE = True` first (500 trials). Full dev is ~29k trials and takes longer.